# Explorer 06 - Networks and communities

**Purpose.** Build the collaboration, credit, and artist-label graphs; compute centrality; detect communities with Leiden; compare them with genre tags.

**Feeds main report section:** 5.5 RQ4 - Communities

**Modules used:** `vinyl.networks`

**Tables read:** `release_artists`, `release_credits`, `release_labels`, `release_genres`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [1]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import time

import pandas as pd
from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-06-networks")
from vinyl import networks
from vinyl import labels as label_power

In [2]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_artists = ingest.read_table(spark, snapshot, "release_artists")
release_credits = ingest.read_table(spark, snapshot, "release_credits")
release_labels = ingest.read_table(spark, snapshot, "release_labels")
release_genres = ingest.read_table(spark, snapshot, "release_genres")
release_styles = ingest.read_table(spark, snapshot, "release_styles")
artists = ingest.read_table(spark, snapshot, "artists")      # names for display only
labels = ingest.read_table(spark, snapshot, "labels")

## 1. Edge lists (Spark)

Minimum edge weights come from `config.GRAPH_MIN_WEIGHT`, chosen so each graph fits on the driver (node and edge counts are in the `config` comment). Self-released labels (`Not On Label` and its variants) are dropped from the artist-label graph, otherwise they become one giant hub.

In [3]:
self_released = label_power.self_released_label_ids(labels).cache()
print("self-released label ids dropped from the artist-label graph:", f"{self_released.count():,}")

w = config.GRAPH_MIN_WEIGHT
min_weight = {"collaboration": w["collaboration"], "credit": w["credit"], "artist-label": w["artist_label"]}
collab = networks.collaboration_edges(release_artists, min_weight=min_weight["collaboration"], max_artists_per_release=20).cache()
credit = networks.credit_edges(release_artists, release_credits, roles=config.PRODUCTION_ROLES, min_weight=min_weight["credit"]).cache()
artist_label = networks.artist_label_edges(release_artists, release_labels, min_weight=min_weight["artist-label"], exclude_label_ids=self_released).cache()
for name, edges in [("collaboration", collab), ("credit", credit), ("artist-label", artist_label)]:
    print(f"{name:14s} min_weight {min_weight[name]}  edges {edges.count():,}")

self-released label ids dropped from the artist-label graph: 312,527


collaboration  min_weight 2  edges 151,838


credit         min_weight 3  edges 334,197


artist-label   min_weight 3  edges 336,173


In [4]:
collab.groupBy("weight").count().orderBy("weight").show(20)   # weight distribution -> choose min_weight

+------+-----+
|weight|count|
+------+-----+
|     2|91285|
|     3|27385|
|     4|12973|
|     5| 6502|
|     6| 3931|
|     7| 2487|
|     8| 1761|
|     9| 1144|
|    10|  841|
|    11|  537|
|    12|  487|
|    13|  375|
|    14|  304|
|    15|  243|
|    16|  194|
|    17|  174|
|    18|  133|
|    19|  104|
|    20|  100|
|    21|  102|
+------+-----+
only showing top 20 rows


## 2. Graphs and centrality (driver)

The artist-label graph uses prefixed ids (`a123` / `l123`) because artist and label ids overlap as numbers.

In [5]:
edge_tables = {
    "collaboration": collab.toPandas(),
    "credit": credit.toPandas(),
    "artist-label": networks.prefixed_bipartite(artist_label).toPandas(),
}
graphs = {k: networks.to_igraph(v) for k, v in edge_tables.items()}
for k, g in graphs.items():
    print(f"{k:14s} nodes {g.vcount():>10,}  edges {g.ecount():>10,}")

# Names only for the nodes in the graphs, not all 10 M artists.
names = {
    "collaboration": networks.node_names(artists, ids=graphs["collaboration"].vs["name"]),
    "credit": networks.node_names(artists, ids=graphs["credit"].vs["name"]),
    "artist-label": networks.node_names(artists, labels, prefixed=True, ids=graphs["artist-label"].vs["name"]),
}
centrality = {k: networks.centrality_table(g, betweenness=False) for k, g in graphs.items()}
for k in graphs:
    print(f"\n{k}: top 20 by PageRank")
    display(centrality[k].nlargest(20, "pagerank").merge(names[k], on="node", how="left"))

collaboration  nodes    113,631  edges    151,838
credit         nodes    209,487  edges    334,197
artist-label   nodes    289,002  edges    336,173



collaboration: top 20 by PageRank


,node,degree,strength,pagerank,name
0,95537,823,3982.0,0.001220,Johann Sebastian Bach
1,95544,831,4663.0,0.001170,Ludwig van Beethoven
2,95546,850,3720.0,0.000990,Wolfgang Amadeus Mozart
3,304975,544,2624.0,0.000669,Johannes Brahms
4,283469,381,1641.0,0.000448,Franz Schubert
5,578727,337,1645.0,0.000426,Robert Schumann
6,999914,300,1639.0,0.000375,Pyotr Ilyich Tchaikovsky
7,260744,199,1855.0,0.000366,Berliner Philharmoniker
8,96123,282,1462.0,0.000366,Claude Debussy
9,216140,282,1467.0,0.000357,Maurice Ravel



credit: top 20 by PageRank


,node,degree,strength,pagerank,name
0,499136,528,2941.0,0.000712,Beau Thomas
1,740272,592,3827.0,0.000701,Brad Boatright
2,495765,520,2588.0,0.000694,Will Killingsworth
3,393350,504,4145.0,0.000561,Matt Colton
4,436674,482,2803.0,0.000541,Carl Saff
5,12544,427,2765.0,0.000515,James Plotkin
6,1124645,235,6138.0,0.000507,Taylor Swift
7,244135,493,3933.0,0.000473,Greg Calbi
8,607445,338,2096.0,0.000445,Mikey Young
9,259758,395,3873.0,0.000434,Ted Jensen



artist-label: top 20 by PageRank


,node,degree,strength,pagerank,name
0,l25487,2403,23875.0,0.002031,Sony Music
1,l1866,939,12231.0,0.000915,Columbia
2,l7703,528,8556.0,0.000758,Deutsche Grammophon
3,l38404,991,8708.0,0.000727,Universal Music Group
4,l23694,1100,7500.0,0.000713,Universal Music
5,l681,583,9264.0,0.000604,Atlantic
6,l1003,643,7839.0,0.000601,BMG
7,l8030,599,3431.0,0.000595,Naxos
8,l5320,546,6095.0,0.000561,Decca
9,l895,508,7306.0,0.000506,RCA


## 3. Leiden communities

In [6]:
communities = {k: networks.leiden_communities(g, resolution=1.0, seed=42) for k, g in graphs.items()}
for k, c in communities.items():
    sizes = c["community"].value_counts()
    print(f"{k:14s} communities {len(sizes):>8,}  >=50 nodes {(sizes >= 50).sum():>5,}  largest {sizes.iloc[0]:>8,}  modularity {c.attrs['modularity']:.4f}")

collaboration  communities   23,939  >=50 nodes    84  largest    4,870  modularity 0.9282
credit         communities   17,057  >=50 nodes    78  largest   10,436  modularity 0.8251
artist-label   communities   38,390  >=50 nodes   130  largest    8,914  modularity 0.9183


## 4. Do communities match genre and style tags?

Each artist gets the genre (and the style) on most of their releases. The credit graph also holds credited people who are never a main artist; they have no tag, so agreement there covers only the nodes that have one. In the artist-label graph agreement covers the artist nodes only.

In [7]:
# Dominant tags only for artists that are in a graph, to keep the driver table small.
artist_ids = sorted(
    set(networks.graph_artist_ids(graphs["collaboration"]))
    | set(networks.graph_artist_ids(graphs["credit"]))
    | set(networks.graph_artist_ids(graphs["artist-label"], prefixed=True))
)
in_graphs = spark.createDataFrame([(i,) for i in artist_ids], "artist_id long")
graph_artists = release_artists.join(in_graphs, "artist_id")
dominant = {
    "genre": networks.artist_dominant_tag(graph_artists, release_genres, "genre").toPandas().rename(columns={"artist_id": "node"}),
    "style": networks.artist_dominant_tag(graph_artists, release_styles, "style").toPandas().rename(columns={"artist_id": "node"}),
}
# Artist-label nodes are prefixed, so prefix the tag table too (label nodes then have no tag).
node_tags = {
    (k, tag): (t.assign(node="a" + t["node"].astype(str)) if k == "artist-label" else t)
    for k in graphs for tag, t in dominant.items()
}

rows = []
for k, g in graphs.items():
    n_artist_nodes = len(networks.graph_artist_ids(g, prefixed=(k == "artist-label")))
    for tag in ("genre", "style"):
        agreement_row = networks.community_tag_agreement(communities[k], node_tags[(k, tag)], tag_col=tag)
        rows.append({"graph": k, "tag": tag, "n_artist_nodes": n_artist_nodes, **agreement_row})
agreement = pd.DataFrame(rows)
agreement["share_tagged"] = agreement["n_nodes"] / agreement["n_artist_nodes"]
display(agreement)

,graph,tag,n_artist_nodes,n_nodes,nmi,ari,purity,share_tagged
0,collaboration,genre,113631,113631,0.304533,0.030042,0.870651,1.000000
1,collaboration,style,113631,109557,0.564332,0.144510,0.604599,0.964147
2,credit,genre,209487,108848,0.232732,0.039772,0.667794,0.519593
3,credit,style,209487,104470,0.381442,0.056730,0.315842,0.498694
4,artist-label,genre,203212,203212,0.277409,0.037212,0.808884,1.000000
5,artist-label,style,203212,197029,0.486645,0.141251,0.440894,0.969574


## 5. Largest communities

Top 10 communities per graph with at least 50 nodes: the five highest-PageRank nodes, the three most common tags and the tag purity (share of the community's tagged nodes that carry the most common tag).

In [8]:
for k in graphs:
    for tag in ("genre", "style"):
        desc = networks.describe_communities(
            communities[k], centrality[k], names[k], node_tags=node_tags[(k, tag)],
            top_n=5, min_size=50, tag_col=tag,
        ).head(10)
        print(f"\n{k}: top 10 communities, dominant {tag}")
        display(desc)


collaboration: top 10 communities, dominant genre


,community,size,top_nodes,n_tagged,top_tags,tag_purity
0,8,4870,"Ludwig van Beethoven, Wolfgang Amadeus Mozart,...",4870,"Classical (0.96), Stage & Screen (0.02), Jazz ...",0.959754
1,13,3627,"Armin van Buuren, Alok (4), Steve Aoki, Dimitr...",3627,"Electronic (0.88), Pop (0.07), Hip Hop (0.01)",0.881720
2,1,3433,"Dj T.H., Talla 2XLC, RAM (38), Christina Novel...",3433,"Electronic (0.99), Pop (0.01), Stage & Screen ...",0.987475
3,7,3227,"Ernesto Rodrigues, Guilherme Rodrigues, Bill L...",3227,"Jazz (0.60), Electronic (0.23), Classical (0.07)",0.599628
4,4,2991,"David Guetta, Ty Dolla $ign, Diplo, French Mon...",2991,"Hip Hop (0.40), Electronic (0.24), Pop (0.16)",0.403210
5,14,2768,"Agathocles, Radek Kopel, Vitimas Do Crack, The...",2768,"Electronic (0.50), Rock (0.49), Non-Music (0.01)",0.496749
6,6,2516,"DJ Spen, Todd Terry, Green Velvet, Robert Owen...",2516,"Electronic (0.91), Funk / Soul (0.04), Pop (0.01)",0.911765
7,27,2037,"Johann Sebastian Bach, Antonio Vivaldi, Georg ...",2037,"Classical (0.98), Jazz (0.01), Folk, World, & ...",0.981836
8,34,1779,"Alchemist, DJ Muggs, Conway The Machine, Macha...",1779,"Hip Hop (0.88), Electronic (0.06), Funk / Soul...",0.879146
9,40,1756,"MC Alee, Dr. Peacock, Deadly Guns, Andy The Co...",1756,"Electronic (0.98), Hip Hop (0.01), Pop (0.01)",0.984055



collaboration: top 10 communities, dominant style


,community,size,top_nodes,n_tagged,top_tags,tag_purity
0,8,4870,"Ludwig van Beethoven, Wolfgang Amadeus Mozart,...",4844,"Romantic (0.28), Modern (0.21), Classical (0.12)",0.277250
1,13,3627,"Armin van Buuren, Alok (4), Steve Aoki, Dimitr...",3601,"House (0.27), Electro House (0.19), Progressiv...",0.269092
2,1,3433,"Dj T.H., Talla 2XLC, RAM (38), Christina Novel...",3430,"Trance (0.62), Progressive Trance (0.12), Prog...",0.619242
3,7,3227,"Ernesto Rodrigues, Guilherme Rodrigues, Bill L...",3195,"Free Improvisation (0.36), Experimental (0.18)...",0.357433
4,4,2991,"David Guetta, Ty Dolla $ign, Diplo, French Mon...",2745,"Trap (0.18), House (0.09), Contemporary R&B (0...",0.181421
5,14,2768,"Agathocles, Radek Kopel, Vitimas Do Crack, The...",2767,"Noise (0.31), Goregrind (0.18), Grindcore (0.12)",0.306469
6,6,2516,"DJ Spen, Todd Terry, Green Velvet, Robert Owen...",2510,"House (0.53), Tech House (0.10), Deep House (0...",0.533068
7,27,2037,"Johann Sebastian Bach, Antonio Vivaldi, Georg ...",2026,"Baroque (0.74), Classical (0.05), Choral (0.04)",0.742843
8,34,1779,"Alchemist, DJ Muggs, Conway The Machine, Macha...",1509,"Boom Bap (0.35), Hardcore Hip-Hop (0.14), Cons...",0.353214
9,40,1756,"MC Alee, Dr. Peacock, Deadly Guns, Andy The Co...",1755,"Hardcore (0.41), Hardstyle (0.39), Happy Hardc...",0.414245



credit: top 10 communities, dominant genre


,community,size,top_nodes,n_tagged,top_tags,tag_purity
0,7,10436,"Taylor Swift, Céline Dion, Madonna, Britney Sp...",3337,"Electronic (0.34), Pop (0.22), Hip Hop (0.16)",0.338328
1,28,7677,"Eminem, Joe LaPorta, Westside Gunn, Pink Siifu...",3315,"Hip Hop (0.83), Electronic (0.08), Funk / Soul...",0.829261
2,15,7211,"Patrick W. Engel, Daniel Krieger, Magnus Lindb...",3373,"Rock (0.84), Electronic (0.07), Folk, World, &...",0.841684
3,25,6781,"Brad Boatright, Will Killingsworth, James Plot...",4108,"Rock (0.86), Electronic (0.09), Folk, World, &...",0.859542
4,1,6575,"Ted Jensen, Chris Bellman, Howie Weinberg, Met...",2366,"Rock (0.75), Electronic (0.06), Pop (0.05)",0.751479
5,4,6504,"Greg Calbi, Heba Kadry, Josh Bonati, Adam Gons...",3261,"Rock (0.50), Electronic (0.18), Folk, World, &...",0.495554
6,22,5996,"Matt Colton, Depeche Mode, Björk, Gorillaz, Jo...",3021,"Rock (0.36), Electronic (0.32), Pop (0.09)",0.364449
7,21,5940,"The Rolling Stones, Bob Ludwig, David Bowie, U...",2207,"Rock (0.49), Electronic (0.14), Folk, World, &...",0.492977
8,10,5674,"Bernie Grundman, Rudy Van Gelder, Kevin Gray, ...",2227,"Jazz (0.32), Rock (0.24), Funk / Soul (0.18)",0.318366
9,0,5127,"Beau Thomas, Simon Davey, Árni Grétar, Michell...",3745,"Electronic (0.92), Rock (0.03), Hip Hop (0.01)",0.920694


... (2 outputs left out)


artist-label: top 10 communities, dominant genre


,community,size,top_nodes,n_tagged,top_tags,tag_purity
0,33,8914,"Transcending Obscurity Records, Nuclear War No...",7373,"Rock (0.87), Electronic (0.12), Folk, World, &...",0.866676
1,25,7190,"Armada (4), Artist Intelligence Agency, Monste...",5907,"Electronic (0.96), Pop (0.02), Hip Hop (0.01)",0.959878
2,18,6792,"Deutsche Grammophon, Naxos, Decca, Sony Classi...",6207,"Classical (0.93), Jazz (0.02), Pop (0.01)",0.932657
3,7,6773,"Universal Music Group, Universal Music, Polydo...",4303,"Pop (0.33), Rock (0.21), Hip Hop (0.17)",0.334650
4,2,6585,"High Roller Records, Nuclear Blast, Napalm Rec...",5301,"Rock (0.95), Electronic (0.02), Folk, World, &...",0.948123
5,3,6341,"Zegema Beach Records, No Time Records, Vægtløs...",4353,"Rock (0.91), Electronic (0.04), Hip Hop (0.01)",0.908109
6,20,5517,"Sony Music, Columbia, RCA, Epic, Music On Vinyl",3734,"Pop (0.31), Rock (0.23), Hip Hop (0.13)",0.306106
7,14,5032,"Empire, BarsOverBs, Copenhagen Crates, Chopped...",3477,"Hip Hop (0.88), Electronic (0.05), Funk / Soul...",0.880069
8,8,4968,"Nervous Records, YHV Records, Quantize Recordi...",4087,"Electronic (0.93), Funk / Soul (0.04), Jazz (0...",0.934426
9,9,4382,"Solid Records (6), Blue Note, ECM Records, Ver...",3472,"Jazz (0.58), Funk / Soul (0.10), Rock (0.09)",0.578053



artist-label: top 10 communities, dominant style


,community,size,top_nodes,n_tagged,top_tags,tag_purity
0,33,8914,"Transcending Obscurity Records, Nuclear War No...",7372,"Black Metal (0.48), Death Metal (0.19), Dungeo...",0.478974
1,25,7190,"Armada (4), Artist Intelligence Agency, Monste...",5872,"House (0.25), Electro House (0.15), Progressiv...",0.247956
2,18,6792,"Deutsche Grammophon, Naxos, Decca, Sony Classi...",6163,"Romantic (0.19), Baroque (0.19), Contemporary ...",0.190654
3,7,6773,"Universal Music Group, Universal Music, Polydo...",4065,"Pop Rock (0.06), Trap (0.05), Indie Pop (0.05)",0.062977
4,2,6585,"High Roller Records, Nuclear Blast, Napalm Rec...",5280,"Heavy Metal (0.21), Black Metal (0.12), Death ...",0.206061
5,3,6341,"Zegema Beach Records, No Time Records, Vægtløs...",4316,"Punk (0.16), Hardcore (0.15), Grindcore (0.10)",0.162651
6,20,5517,"Sony Music, Columbia, RCA, Epic, Music On Vinyl",3525,"Pop Rock (0.06), Alternative Rock (0.05), Ball...",0.061560
7,14,5032,"Empire, BarsOverBs, Copenhagen Crates, Chopped...",2966,"Boom Bap (0.29), Instrumental (0.14), Consciou...",0.293999
8,8,4968,"Nervous Records, YHV Records, Quantize Recordi...",4079,"House (0.49), Tech House (0.13), Deep House (0...",0.489826
9,9,4382,"Solid Records (6), Blue Note, ECM Records, Ver...",3372,"Contemporary Jazz (0.20), Bop (0.05), Post Bop...",0.200771


## 6. Resolution sweep

How stable are the communities when Leiden's resolution changes? `nmi_vs_reference` compares each run with resolution 1.0. The credit graph is only swept if the time estimated from the collaboration run (scaled by edge count) is under 5 minutes.

In [9]:
sweeps = {}
start = time.perf_counter()
sweeps["collaboration"] = networks.resolution_sweep(graphs["collaboration"], resolutions=(0.5, 1.0, 2.0), seed=42, reference=1.0)
collab_seconds = time.perf_counter() - start
print(f"collaboration sweep: {collab_seconds:.1f} s")
display(sweeps["collaboration"])

estimate = collab_seconds * graphs["credit"].ecount() / graphs["collaboration"].ecount()
print(f"estimated credit sweep: {estimate:.0f} s (limit 300 s)")
if estimate < 300:
    start = time.perf_counter()
    sweeps["credit"] = networks.resolution_sweep(graphs["credit"], resolutions=(0.5, 1.0, 2.0), seed=42, reference=1.0)
    print(f"credit sweep: {time.perf_counter() - start:.1f} s")
    display(sweeps["credit"])
else:
    print("credit sweep skipped")

collaboration sweep: 110.1 s


,resolution,n_communities,n_communities_ge50,largest_share,modularity,nmi_vs_reference
0,0.5,23864,57,0.068080,0.920419,0.966256
1,1.0,23939,84,0.042858,0.928191,1.000000
2,2.0,24036,118,0.028091,0.921134,0.963483


estimated credit sweep: 242 s (limit 300 s)


credit sweep: 421.1 s


,resolution,n_communities,n_communities_ge50,largest_share,modularity,nmi_vs_reference
0,0.5,17242,50,0.140338,0.794122,0.835816
1,1.0,17057,78,0.049817,0.825073,1.000000
2,2.0,17099,139,0.026212,0.814004,0.850401


## 7. Summary table

One row per graph and tag type. The report quotes this table.

In [10]:
graph_stats = pd.DataFrame({
    "graph": list(graphs),
    "min_weight": [min_weight[k] for k in graphs],
    "nodes": [g.vcount() for g in graphs.values()],
    "edges": [g.ecount() for g in graphs.values()],
    "communities": [communities[k]["community"].nunique() for k in graphs],
    "modularity": [communities[k].attrs["modularity"] for k in graphs],
})
summary = graph_stats.merge(agreement, on="graph")
display(summary[["graph", "min_weight", "nodes", "edges", "communities", "modularity", "tag", "n_nodes", "share_tagged", "nmi", "ari", "purity"]].round(4))

,graph,min_weight,nodes,edges,communities,modularity,tag,n_nodes,share_tagged,nmi,ari,purity
0,collaboration,2,113631,151838,23939,0.9282,genre,113631,1.0000,0.3045,0.0300,0.8707
1,collaboration,2,113631,151838,23939,0.9282,style,109557,0.9641,0.5643,0.1445,0.6046
2,credit,3,209487,334197,17057,0.8251,genre,108848,0.5196,0.2327,0.0398,0.6678
3,credit,3,209487,334197,17057,0.8251,style,104470,0.4987,0.3814,0.0567,0.3158
4,artist-label,3,289002,336173,38390,0.9183,genre,203212,1.0000,0.2774,0.0372,0.8089
5,artist-label,3,289002,336173,38390,0.9183,style,197029,0.9696,0.4866,0.1413,0.4409


## Decisions to carry into `main.ipynb`

Fill this in when the exploration is done. Each line should end up as a sentence in the report's methodology or limitations.

- [ ] `min_weight` and `max_artists_per_release` for each graph, and resulting node/edge counts.
- [ ] Leiden resolution, seed, and modularity.
- [ ] Agreement scores (NMI, ARI, purity) against genre and against style.
- [ ] Names for the communities shown in the report.